# 18 — Một giao thức thống nhất: Validation và Test

## Input → Method → Experiment → Results → Discussion

**Mục tiêu:** mọi mô hình thành viên học trên cùng 3.792 mẫu. Tập 702 mẫu nội bộ dùng chọn epoch; C=642 dùng calibration; M=642 dùng học Ensemble. Giữ assignment theo nhóm và seed 42. Không dùng H để fit hoặc chọn tham số.

**Chọn checkpoint neural:** cả |Train loss − Validation loss| < 0,1 và |Train Weighted-F1 − Validation Weighted-F1| < 0,1. Trong checkpoint đạt điều kiện, chọn F1 nội bộ cao nhất; hòa thì chọn loss thấp hơn rồi epoch sớm hơn. Đo hai tập trong eval mode, tắt dropout. ML dùng cấu hình cố định, không có epoch; báo gap thực, không dựng đường cong giả. Gap nhỏ chưa chứng minh không overfit.

**Mô hình:** tái sử dụng GCN, GAT, BERT-Base, RoBERTa-Base, GPT-2 đã fit đúng 3.792 mẫu. Fit lại Word LR, LinearSVC, Word NB, Char LR, BERT-Tiny và DistilBERT theo cùng phạm vi dữ liệu. Ngân sách kiến trúc khác nhau được công bố trong config. Không refit toàn A hoặc Train-full, không multi-seed/5-fold.

**Xác suất và Ensemble:** sigmoid cố định fit chỉ trên C. B0 = NB + Char LR; B1 = SVM + Char LR; B2 = NB + Char LR + SVM; B3 = SVM + DistilBERT. So sánh Equal Mean, Weighted Mean, nonnegative 2-additive Choquet; normalized pair-min cho cặp hai thành viên. Tham số pooling học chỉ trên M theo log loss, SLSQP và ràng buộc đã khóa. Không tuyên bố thuật toán mới.

**Đánh giá cuối:** khóa toàn bộ checkpoint, calibrator và combiner trước Validation 2.139 và Test 2.140. Validation loại đúng một exact duplicate đã duyệt; Test giữ nguyên từ mirror được duyệt. Test đã được xem ở đợt trước: đây là đánh giá lại tập đã biết, không phải kiểm chứng độc lập mới. Không chọn hoặc thay tham số bằng điểm Validation/Test.

**Đầu ra:** đầy đủ metric, confusion matrix, prediction theo ID, calibration bins, config, checkpoint, checksum, log và thời gian. Một seed chưa chứng minh ổn định. Dữ liệu gốc giữ nguyên; chỉ app, hình và các bản thừa đã xác minh được dọn theo quyền người dùng.


## 00 — Kiểm tra đầu vào và khóa giao thức
**Đầu vào:** assignment, tài nguyên và checkpoint. **Xử lý:** kiểm SHA256, nhóm, ID, số mẫu và khóa config trước training. **Đầu ra:** preflight cùng năm base lock tái sử dụng. **Kiểm tra:** không đọc Validation/Test trước global lock.


In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
assert ROOT==Path('C:/Users/npd20/Downloads/CVDHTTTNT_Duy')
sys.path.insert(0,str(ROOT/'src'))
import unified_coursework as experiment
experiment.prepare()
experiment.reuse()


## 01 — Bốn baseline ML
**Đầu vào:** Train nội bộ 3.792 mẫu. **Xử lý:** fit vocabulary và classifier chỉ trên Train, dùng siêu tham số gốc, không grid search. **Đầu ra:** bốn pipeline và checksum. **Kiểm tra:** hội tụ, thứ tự nhãn, thời gian; không dùng tập đánh giá để fit.


In [ ]:
experiment.fit_classical()


## 02 — BERT-Tiny
**Đầu vào:** pretrained Google đã xác minh, cùng Train/Validation nội bộ. **Xử lý:** tối đa 4 epoch, batch16, lr5e-5, dropout0.2, weight_decay0.1, max128. **Đầu ra:** history, checkpoint được chọn và lock. **Kiểm tra:** hai gap, finite; không thay bằng checkpoint Train-full cũ.


In [ ]:
experiment.fit_neural('bert_tiny')


## 03 — DistilBERT
**Đầu vào:** pretrained publisher đã xác minh, cùng 3.792/702 mẫu. **Xử lý:** tối đa 3 epoch, batch4, lr1e-5; các thiết lập chung trong config. **Đầu ra:** history, checkpoint và lock. **Kiểm tra:** cả hai gap; tối đa 60 phút, không tự nới điều kiện.


In [ ]:
experiment.fit_neural('distilbert')


## 04 — Calibration trên C và prediction nội bộ
**Đầu vào:** 11 checkpoint cùng giao thức, C/M tách nhóm. **Xử lý:** sigmoid fit chỉ trên C; dự đoán M để học combiner; đo Train/Validation nội bộ để chẩn đoán. **Đầu ra:** calibrator, metric, prediction, bins và thời gian. **Kiểm tra:** không gọi điểm M fit là đánh giá độc lập. Đã thêm giải phóng cache GPU sau khi preflight gặp thiếu VRAM; không huấn luyện lại mô hình.


In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
experiment.calibrate()

## 05 — Học combiner trên M và khóa toàn bộ tham số
**Đầu vào:** calibrated M prediction cùng sample ID. **Xử lý:** học các cấu hình đã chốt; kiểm trọng số không âm và tổng bằng 1. **Đầu ra:** combiner parameters và global_development_lock. **Kiểm tra:** khóa trước khi đọc Validation/Test; không chọn combiner bằng điểm đánh giá.


In [ ]:
lock=experiment.meta_and_freeze()
print(lock['status'])


## 06 — Đánh giá Validation 2.139 và Test 2.140
**Đầu vào:** frozen parameters và nguồn dữ liệu đã duyệt. **Xử lý:** xác minh checksum/Git blob, ID và văn bản khớp nhãn; inference và tính metric, không tuning. **Đầu ra:** metric, prediction, confusion counts, bins, timing, EVALUATION_COMPLETE. **Kiểm tra:** mọi hàng thuộc đúng checkpoint/giao thức; ghi rõ Test đã được xem trước đó.


In [ ]:
final_metrics=experiment.evaluate_final()


## 07 — Kết quả và thảo luận
**Đầu vào:** final_metrics đã lưu. **Xử lý:** trình bày đầy đủ metric theo split. **Đầu ra:** bảng thật cho báo cáo, app và hình. **Kiểm tra:** không xếp hạng chéo giao thức cũ, không gọi single-seed là ổn định, không chọn tham số bằng Test. Giữ kết quả âm và các giới hạn.


In [ ]:
from IPython.display import display
display(final_metrics[['model','split','n','weighted_f1','macro_f1','accuracy','precision_weighted','recall_weighted','balanced_accuracy','mcc','log_loss','brier_real_binary']])
